# AIFS Single v2: comprensión y validación mínima


> Es un prototipo de investigación. No pretende ser todavía el programa de producción 

## Fuentes

- Notebook oficial:  
  `https://huggingface.co/ecmwf/aifs-single-2.0/resolve/main/run_AIFS_v2.0.ipynb`
- Modelo:  
  `https://huggingface.co/ecmwf/aifs-single-2.0`
- Script de AIFS ENS de Milton `.\scripts\block1\legacy\state_runner_altMARS`

El notebook oficial y los scripts de ECMWF se publican bajo Apache 2.0; los pesos del modelo, bajo CC BY 4.0. Conserva la atribución al reutilizar el material.

## 0. Diferencias respecto al script de ensamble

| Elemento | Flujo ENS de referencia | Este notebook |
|---|---|---|
| Checkpoint | `ecmwf/aifs-ens-1.0` | `ecmwf/aifs-single-2.0` |
| Miembros | dimensión y bucle `member` | pronóstico determinista |
| Configuración | `argparse` | `NotebookConfig` |
| Variables | atmósfera/superficie | añade oleaje, nieve y 10 hPa |
| Entrada temporal | \(t-6\) h y \(t_0\) | \(t-6\) h y \(t_0\) |
| Salida inicial | NetCDF global grande | subconjunto regional pequeño |
| Horizonte  | varios días | hasta 72 h  |



## 1. Dependencias

El ejemplo oficial está orientado a Python 3.11/3.12 y GPU NVIDIA Ampere o posterior. La combinación CUDA–PyTorch–FlashAttention debe ser compatible con el sistema.

Descomenta estas líneas únicamente en un entorno o kernel dedicado.

In [ ]:
# %pip install "anemoi-inference[huggingface]==0.8.3" \
#              "anemoi-models==0.9.3" \
#              "anemoi-utils==0.4.35.post3"
# %pip install "torch==2.7.0" "torch-geometric==2.6.1"
# %pip install "earthkit-regrid==0.5.1" \
#              "ecmwf-opendata==0.3.29" \
#              "earthkit-data<1.0.0"
# %pip install "flash-attn==2.7.4.post1"
# %pip install xarray netCDF4 matplotlib cartopy pandas
#
# Reinicia el kernel después de instalar.

## 2. Variables de entorno

Deben establecerse antes de importar PyTorch y Anemoi.

- `PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True`: ayuda a reducir fragmentación de memoria.
- `ANEMOI_INFERENCE_NUM_CHUNKS`: controla particiones internas; valores mayores suelen reducir memoria a costa de tiempo.

In [1]:
import os

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
os.environ.setdefault("ANEMOI_INFERENCE_NUM_CHUNKS", "16")

'16'

## 3. Importaciones

In [2]:
import datetime as dt
import platform
from collections import defaultdict
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Iterable, Mapping

import numpy as np
import pandas as pd
import xarray as xr
import torch

import earthkit.data as ekd
import earthkit.regrid as ekr

from anemoi.inference.outputs.printer import print_state
from anemoi.inference.runners.simple import SimpleRunner
from IPython.display import display

## 4. Configuración del experimento

En Jupyter se usa un objeto de configuración en lugar de `argparse`. Esto evita el argumento interno:

```text
-f .../jupyter/runtime/kernel-....json
```

El modo seguro está activado por defecto:

- `run_download=False`: no descarga varios gigabytes.
- `run_inference=False`: no carga ni ejecuta el modelo.

Para una prueba completa, primero activa la descarga; después de validar `input_state`, activa la inferencia.

In [3]:
@dataclass(frozen=True)
class NotebookConfig:
    # Backend de datos. Para este flujo debe ser "mars".
    data_source: str = "mars"

    # Fecha inicial t0 del pronóstico. MARS no tiene un equivalente robusto
    # a latest() en este notebook; por ello se requiere una fecha explícita.
    # Ejemplo: "2020-10-05T18:00:00Z"
    date: str | None = "2020-10-05T18:00:00Z"

    # Configuración MARS para análisis operacionales archivados.
    # Ajusta mars_class si tu acceso usa otra clase, por ejemplo "ea" para ERA5.
    mars_class: str = "od"
    mars_type: str = "an"
    mars_stream: str = "oper"
    mars_expver: str = "1"
    mars_grid: str = "N320"

    # Modo de solicitud. Por defecto se conserva la semántica de la
    # función MARS que ya funcionaba para superficie.
    mars_source: str = "ecmwf"
    mars_request_mode: str = "legacy_kwargs"
    mars_use_class_keyword: bool = False
    mars_use_expver_keyword: bool = False

    # lead_time_hours: intervalo de pronóstico deseado.
    lead_time_hours: int = 6
    device: str = "cuda"

    run_download: bool = True
    run_inference: bool = True

    output_variables: tuple[str, ...] = (
        "msl", "10u", "10v", "t_850"
    )

    # México, Golfo de México y mares adyacentes.
    # Longitudes en 0–360.
    lat_min: float = 5.0
    lat_max: float = 35.0
    lon_min: float = 240.0
    lon_max: float = 300.0

    output_dir: Path = Path("./outputs/aifs_single_v2")


CONFIG = NotebookConfig()
display(pd.Series(asdict(CONFIG), name="value").to_frame())


,value
data_source,mars
date,2020-10-05T18:00:00Z
mars_class,od
mars_type,an
mars_stream,oper
mars_expver,1
mars_grid,N320
mars_source,ecmwf
mars_request_mode,legacy_kwargs
mars_use_class_keyword,False


## 5. Diagnóstico del entorno

Este es el contenido general de las condiciones para correr AIFS Single 2.0 en el que se establece las condiciones para ejecutar 

In [4]:
print("Plataforma:", platform.platform())
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA disponible:", torch.cuda.is_available())

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    major, minor = torch.cuda.get_device_capability(0)
    print("GPU:", gpu_name)
    print(f"Compute capability: {major}.{minor}")

    if CONFIG.run_inference and major < 8:
        raise RuntimeError(
            "La prueba oficial requiere Ampere o posterior "
            "(compute capability >= 8.0)."
        )
elif CONFIG.run_inference:
    raise RuntimeError(
        "run_inference=True, pero PyTorch no detecta CUDA."
    )
else:
    print(
        "La GPU no es necesaria para inspeccionar la lógica; "
        "sí será necesaria para la inferencia."
    )

try:
    import flash_attn
    print("FlashAttention:", flash_attn.__version__)
except ImportError:
    if CONFIG.run_inference:
        raise
    print(
        "FlashAttention no está disponible. Es aceptable para "
        "inspección, pero no para la inferencia oficial."
    )

Plataforma: Linux-5.14.0-427.37.1.el9_4.x86_64-x86_64-with-glibc2.36
Python: 3.12.7
PyTorch: 2.7.0+cu126
CUDA disponible: True
GPU: NVIDIA H100
Compute capability: 9.0
FlashAttention: 2.7.4.post1


In [4]:
def parse_utc_datetime(value: str) -> dt.datetime:
    """
    Convierte una fecha ISO a ``datetime`` UTC sin zona horaria.

    La fecha se redondea al inicio de la hora porque MARS separa la fecha
    de análisis en los keywords ``date`` y ``time``.
    """

    parsed = dt.datetime.fromisoformat(value.replace("Z", "+00:00"))
    if parsed.tzinfo is not None:
        parsed = parsed.astimezone(dt.timezone.utc).replace(tzinfo=None)
    return parsed.replace(minute=0, second=0, microsecond=0)


def resolve_initial_date(config: NotebookConfig) -> dt.datetime:
    """
    Define y valida la fecha inicial del pronóstico.

    Para MARS se exige una fecha explícita. Esto evita mezclar el flujo de
    ECMWF Open Data, donde existe ``latest()``, con el archivo MARS, donde la
    disponibilidad depende de clase, stream, permisos y retención.
    """

    if config.date is None:
        raise ValueError(
            "CONFIG.date debe definirse explícitamente para MARS, por ejemplo "
            "'2020-10-05T18:00:00Z'."
        )

    resolved = parse_utc_datetime(config.date)

    if resolved.hour not in {0, 6, 12, 18}:
        raise ValueError(
            f"{resolved.hour:02d} UTC no es una hora sinóptica válida. "
            "Usa 00, 06, 12 o 18 UTC."
        )

    return resolved


DATE = resolve_initial_date(CONFIG)
print(f"Valor de la fecha desde la configuración inicial: {CONFIG.date}")
print(f"Valor de la fecha en formato ISO: {DATE}")

PREVIOUS_DATE = DATE - dt.timedelta(hours=6)

print("t-6 h:", PREVIOUS_DATE)
print("t0   :", DATE)


Valor de la fecha desde la configuración inicial: 2020-10-05T18:00:00Z
Valor de la fecha en formato ISO: 2020-10-05 18:00:00
t-6 h: 2020-10-05 12:00:00
t0   : 2020-10-05 18:00:00


## 7. Inventario de variables de AIFS Single v2

La versión 2 añade variables de oleaje, nieve y el nivel de 10 hPa. La humedad específica en 10 y 50 hPa se retira de la entrada prognóstica en el flujo oficial.

In [5]:
PARAM_SFC = [
    "10u", "10v", "2d", "2t", "msl", "skt", "sp",
    "tcw", "lsm", "z", "slor", "sdor", "sd",
]

PARAM_SOIL = ["vsw", "sot"]
# Convenio alternativo usado por el archivo operacional MARS en superficie.
# Si levtype="sol" con vsw/sot no existe, se recuperan directamente estas
# variables como campos single-level: swvl1, swvl2, stl1, stl2.
PARAM_SOIL_SFC = ["swvl1", "swvl2", "stl1", "stl2"]

PARAM_WAVE = [
    "wmb", "h1012", "h1214", "h1417", "h1721",
    "h2125", "h2530", "mwd", "cdww", "mwp", "swh",
]

PARAM_PL = ["z", "t", "u", "v", "q"]

LEVELS = [
    1000, 925, 850, 700, 600, 500, 400,
    300, 250, 200, 150, 100, 50, 10,
]

SOIL_LEVELS = [1, 2]

## 8. Recuperación desde MARS

Para AIFS Single v2 se requieren dos análisis consecutivos: \(t-6\ \mathrm{h}\) y \(t_0\). En el flujo con MARS no se usa `ecmwf-open-data`, no se desplazan longitudes y no se interpola desde una grilla regular de 0.25°. La solicitud se hace directamente con `earthkit.data.from_source("mars", request=...)` y `grid="N320"` para obtener la grilla reducida gaussiana que espera el modelo.

El punto importante es que MARS no infiere automáticamente todos los keywords. Deben declararse de forma explícita `class`, `type`, `stream`, `expver`, `date`, `time`, `param`, `levtype`, `levelist` cuando aplique, y `grid`.


### LECTURA DE CONDICIONES INICIALES: MARS

La recuperación se centraliza en `get_data`. Esta función devuelve el mismo contrato que esperaba el resto del notebook: un diccionario `{nombre_variable: array}` con arreglos de forma `(2, n_points)`, donde el eje temporal corresponde a `t-6 h` y `t0`.


In [6]:
ekd.config.set({"cache-policy": "user"})


class MarsRetrievalError(RuntimeError):
    """Error enriquecido para solicitudes MARS fallidas."""


def _as_list(value: str | Iterable[str]) -> list[str]:
    """Normaliza parámetros que pueden recibirse como texto o iterable."""

    if isinstance(value, str):
        return [value]
    return list(value)


def _field_name(field, *, use_level: bool) -> str:
    """
    Obtiene un nombre estable para cada campo GRIB.

    Se prioriza `param`, porque coincide con el convenio usado en el notebook
    original de AIFS. Si no está disponible, se usa `shortName`.
    """

    try:
        name = field.metadata("param")
    except Exception:
        name = field.metadata("shortName")

    if use_level:
        try:
            level = field.metadata("levelist")
        except Exception:
            level = field.metadata("level")
        name = f"{name}_{level}"

    return str(name)


def _mars_date_time(valid_date: dt.datetime) -> dict[str, str]:
    """
    Convierte un `datetime` UTC en formato MARS clásico.

    Usamos YYYYMMDD y HHMM para la ruta explícita. En la ruta compatible con
    la función que ya funcionaba se pasa directamente el objeto `datetime`.
    """

    return {
        "date": valid_date.strftime("%Y%m%d"),
        "time": valid_date.strftime("%H%M"),
    }


def _mars_request_payload(
    *,
    valid_date: dt.datetime,
    config: NotebookConfig,
    param: str | Iterable[str],
    levtype: str,
    levelist: Iterable[int] | None = None,
    stream: str | None = None,
    mode: str | None = None,
    **mars_overrides,
) -> tuple[str, dict]:
    """
    Construye la solicitud MARS.

    `mode="legacy_kwargs"` reproduce la semántica de la función que ya
    permitía descargar superficie: `ekd.from_source("mars", date=<datetime>,
    source="ecmwf", ...)`. Esto evita forzar `class` y `expver`, que en tu
    ejecución dieron cero campos incluso para `10u`.

    `mode="explicit_request"` usa un diccionario MARS explícito. Es útil para
    depurar, pero no debe ser el modo por defecto si la ruta antigua ya
    funcionaba para superficie.
    """

    selected_mode = mode or getattr(config, "mars_request_mode", "legacy_kwargs")
    requested_params = _as_list(param)
    requested_levels = list(levelist) if levelist is not None else None

    if selected_mode == "legacy_kwargs":
        payload = {
            "date": valid_date,
            "source": getattr(config, "mars_source", "ecmwf"),
            "param": requested_params,
            "grid": config.mars_grid,
            "levtype": levtype,
            "type": config.mars_type,
            "stream": stream or config.mars_stream,
        }
        if requested_levels is not None:
            payload["levelist"] = requested_levels
        payload.update(mars_overrides)
        return selected_mode, payload

    if selected_mode == "explicit_request":
        request = {
            "type": config.mars_type,
            "stream": stream or config.mars_stream,
            "levtype": levtype,
            "param": requested_params,
            "grid": config.mars_grid,
            **_mars_date_time(valid_date),
        }

        # Estos filtros NO se fuerzan por defecto. En tu caso, `class='od'`
        # + `expver='1'` devolvió cero campos para 10u; por eso se activan
        # sólo si los defines explícitamente en CONFIG.
        if getattr(config, "mars_use_class_keyword", False):
            request["class"] = config.mars_class
        if getattr(config, "mars_use_expver_keyword", False):
            request["expver"] = config.mars_expver
        if requested_levels is not None:
            request["levelist"] = requested_levels

        request.update(mars_overrides)
        return selected_mode, {"request": request}

    raise ValueError(
        "Modo MARS no reconocido. Usa 'legacy_kwargs' o 'explicit_request'."
    )


def _request_label(mode: str, payload: Mapping) -> str:
    """Resume una solicitud MARS para diagnóstico."""

    request = payload.get("request", payload)
    keys = [
        "source", "class", "type", "stream", "expver", "levtype",
        "param", "levelist", "date", "time", "grid",
    ]
    parts = [f"mode={mode!r}"]
    parts.extend(f"{key}={request.get(key)!r}" for key in keys if key in request)
    return ", ".join(parts)


def _retrieve_mars(mode: str, payload: Mapping):
    """Ejecuta la recuperación MARS con la forma requerida por earthkit."""

    if mode == "legacy_kwargs":
        return ekd.from_source("mars", **payload)
    if mode == "explicit_request":
        return ekd.from_source("mars", **payload)
    raise ValueError(f"Modo MARS no reconocido: {mode!r}")


def mars_probe_request(
    *,
    date: dt.datetime,
    config: NotebookConfig,
    params: Iterable[str],
    levtype: str,
    levelist: Iterable[int] | None = None,
    stream: str | None = None,
    max_params: int | None = None,
    mode: str | None = None,
    **mars_overrides,
) -> pd.DataFrame:
    """
    Prueba variables MARS una por una usando el mismo modo que `get_data`.
    """

    rows = []
    selected_params = list(params)
    if max_params is not None:
        selected_params = selected_params[:max_params]

    for parameter in selected_params:
        for valid_date in (date - dt.timedelta(hours=6), date):
            selected_mode, payload = _mars_request_payload(
                valid_date=valid_date,
                config=config,
                param=parameter,
                levtype=levtype,
                levelist=levelist,
                stream=stream,
                mode=mode,
                **mars_overrides,
            )
            label = _request_label(selected_mode, payload)
            try:
                data = _retrieve_mars(selected_mode, payload)
                fieldlist = data.to_fieldlist()
                n_fields = len(fieldlist)
                rows.append(
                    {
                        "param": parameter,
                        "valid_time": valid_date.isoformat(),
                        "levtype": levtype,
                        "stream": stream or config.mars_stream,
                        "levelist": list(levelist) if levelist is not None else None,
                        "n_fields": n_fields,
                        "status": "ok" if n_fields > 0 else "empty",
                        "request": label,
                        "error": "",
                    }
                )
            except Exception as exc:
                rows.append(
                    {
                        "param": parameter,
                        "valid_time": valid_date.isoformat(),
                        "levtype": levtype,
                        "stream": stream or config.mars_stream,
                        "levelist": list(levelist) if levelist is not None else None,
                        "n_fields": 0,
                        "status": "error",
                        "request": label,
                        "error": f"{type(exc).__name__}: {exc}",
                    }
                )

    return pd.DataFrame(rows)


def get_data(
    *,
    date: dt.datetime,
    config: NotebookConfig,
    param: str | Iterable[str],
    levtype: str,
    levelist: Iterable[int] | None = None,
    stream: str | None = None,
    split_by_param: bool = False,
    mode: str | None = None,
    **mars_overrides,
) -> dict[str, np.ndarray]:
    """
    Recupera campos meteorológicos desde MARS para `t-6 h` y `t0`.

    Por defecto se usa `mode="legacy_kwargs"`, que reproduce la llamada MARS
    que previamente sí permitía descargar campos de superficie. Esto mantiene
    la ruta funcional y evita imponer filtros MARS que pueden vaciar la
    consulta para el archivo operativo.
    """

    if config.data_source.lower() != "mars":
        raise ValueError(
            f"Este `get_data` implementa sólo MARS; se recibió "
            f"{config.data_source!r}."
        )

    requested_params = _as_list(param)
    requested_levels = list(levelist) if levelist is not None else None
    collected: defaultdict[str, list[np.ndarray]] = defaultdict(list)

    for valid_date in (date - dt.timedelta(hours=6), date):
        param_batches = [[p] for p in requested_params] if split_by_param else [requested_params]

        for param_batch in param_batches:
            selected_mode, payload = _mars_request_payload(
                valid_date=valid_date,
                config=config,
                param=param_batch,
                levtype=levtype,
                levelist=requested_levels,
                stream=stream,
                mode=mode,
                **mars_overrides,
            )

            try:
                data = _retrieve_mars(selected_mode, payload)
            except Exception as exc:
                raise MarsRetrievalError(
                    "La solicitud MARS no devolvió los campos esperados.\n"
                    f"Request: {_request_label(selected_mode, payload)}\n"
                    "Si superficie funcionaba antes, mantén "
                    "CONFIG.mars_request_mode='legacy_kwargs' y no fuerces "
                    "class/expver.\n"
                    f"Error original: {type(exc).__name__}: {exc}"
                ) from exc

            fieldlist = data.to_fieldlist()
            if len(fieldlist) == 0:
                raise MarsRetrievalError(
                    "MARS devolvió cero campos.\n"
                    f"Request: {_request_label(selected_mode, payload)}"
                )

            for field in fieldlist:
                values = np.asarray(field.to_numpy())
                if values.ndim != 1:
                    values = values.reshape(-1)

                name = _field_name(
                    field,
                    use_level=requested_levels is not None,
                )
                collected[name].append(values)

    expected_ntimes = 2
    output = {}
    for name, time_slices in collected.items():
        if len(time_slices) != expected_ntimes:
            raise ValueError(
                f"{name}: se esperaban {expected_ntimes} tiempos "
                f"(t-6 h y t0), pero se recibieron {len(time_slices)}."
            )

        shapes = {tuple(np.asarray(x).shape) for x in time_slices}
        if len(shapes) != 1:
            raise ValueError(
                f"{name}: las formas espaciales no son consistentes: "
                f"{sorted(shapes)}."
            )

        output[name] = np.stack(time_slices, axis=0)

    return output


### Notas sobre los keywords MARS usados

- `levtype="sfc"`: variables de superficie y oleaje.
- `levtype="sol"`: variables de suelo por niveles `1` y `2`.
- `levtype="pl"`: variables atmosféricas en niveles de presión.
- `stream="wave"`: oleaje; el resto usa `CONFIG.mars_stream`, por defecto `"oper"`.
- `grid="N320"`: salida en grilla reducida gaussiana compatible con AIFS; por eso aquí no se usa `earthkit-regrid` antes de la inferencia.


In [ ]:
# La función get_open_data se eliminó del flujo activo.
# Si necesitas conservar el backend de ECMWF Open Data para pruebas NRT,
# conviene implementarlo como una función separada, no mezclarlo con MARS.


In [7]:
def assert_fields_present(
    fields: Mapping[str, np.ndarray],
    expected: Iterable[str],
    *,
    group_name: str,
) -> None:
    """
    Verifica que un conjunto de campos contenga todas las variables esperadas.
    """

    missing = set(expected) - set(fields)
    if missing:
        raise KeyError(
            f"Faltan variables en {group_name}: {sorted(missing)}"
        )


def retrieve_soil_fields(
    *,
    date: dt.datetime,
    config: NotebookConfig,
) -> dict[str, np.ndarray]:
    """
    Recupera campos de suelo desde MARS operational analysis.

    Primero intenta el convenio GRIB2 por tipo de nivel de suelo:

        levtype="sol", param=["vsw", "sot"], levelist=[1, 2]

    En el archivo operacional usado aquí ese request puede devolver cero campos.
    Por ello se usa como respaldo el convenio single-level:

        levtype="sfc", param=["swvl1", "swvl2", "stl1", "stl2"]

    Ambos caminos retornan un diccionario compatible con
    `transform_initial_conditions`.
    """

    try:
        soil = get_data(
            date=date,
            config=config,
            param=PARAM_SOIL,
            levtype="sol",
            levelist=SOIL_LEVELS,
        )
        expected_soil = [
            f"{parameter}_{level}"
            for parameter in PARAM_SOIL
            for level in SOIL_LEVELS
        ]
        assert_fields_present(soil, expected_soil, group_name="suelo")
        print("Suelo recuperado con levtype='sol' y param=['vsw', 'sot'].")
        return soil

    except MarsRetrievalError as exc:
        print(
            "Aviso: el request de suelo con levtype='sol' no devolvió campos. "
            "Se intentará el convenio operacional single-level "
            "['swvl1', 'swvl2', 'stl1', 'stl2'] con levtype='sfc'."
        )
        print(str(exc).splitlines()[0])

    soil = get_data(
        date=date,
        config=config,
        param=PARAM_SOIL_SFC,
        levtype="sfc",
    )
    assert_fields_present(soil, PARAM_SOIL_SFC, group_name="suelo")
    print("Suelo recuperado con levtype='sfc' y parámetros swvl*/stl*.")
    return soil


def retrieve_raw_initial_conditions(
    *,
    date: dt.datetime,
    config: NotebookConfig,
) -> tuple[dict[str, np.ndarray], dict[str, np.ndarray]]:
    """
    Descarga desde MARS las condiciones iniciales requeridas por AIFS Single v2.

    Retorna dos diccionarios: `fields`, con superficie, oleaje y niveles de
    presión; y `soil`, con los campos de suelo que posteriormente se renombran
    o incorporan al convenio esperado por el modelo.
    """

    fields: dict[str, np.ndarray] = {}

    surface = get_data(
        date=date,
        config=config,
        param=PARAM_SFC,
        levtype="sfc",
    )
    assert_fields_present(surface, PARAM_SFC, group_name="superficie")
    fields.update(surface)

    wave = get_data(
        date=date,
        config=config,
        param=PARAM_WAVE,
        levtype="sfc",
        stream="wave",
    )
    assert_fields_present(wave, PARAM_WAVE, group_name="oleaje")
    fields.update(wave)

    soil = retrieve_soil_fields(
        date=date,
        config=config,
    )

    pressure = get_data(
        date=date,
        config=config,
        param=PARAM_PL,
        levtype="pl",
        levelist=LEVELS,
    )
    expected_pressure = [
        f"{parameter}_{level}"
        for parameter in PARAM_PL
        for level in LEVELS
    ]
    assert_fields_present(
        pressure,
        expected_pressure,
        group_name="niveles de presión",
    )
    fields.update(pressure)

    return fields, soil


### Ejecutar o saltar la descarga

La descarga completa puede tardar y ocupar varios gigabytes de caché. Solo se ejecuta con `CONFIG.run_download=True`.

In [8]:
raw_fields = None
raw_soil = None
if CONFIG.run_download:
    raw_fields, raw_soil = retrieve_raw_initial_conditions(
        date=DATE,
        config=CONFIG,
    )
    print("Campos principales:", len(raw_fields))
    print("Campos de suelo:", len(raw_soil))
else:
    print(
        "Descarga omitida. Activa CONFIG.run_download y ejecuta "
        "de nuevo desde la configuración."
    )


2026-07-02 14:58:52 ECMWF API python library 1.7.0
2026-07-02 14:58:52 ECMWF API at https://api.ecmwf.int/v1
2026-07-02 14:58:52 Welcome Monika Feldmann
2026-07-02 14:58:53 In case of problems, please check https://confluence.ecmwf.int/display/WEBAPI/Web+API+FAQ or contact servicedesk@ecmwf.int
2026-07-02 14:58:53 Request submitted
2026-07-02 14:58:53 Request id: 6a46608de3187990d61d29bc
2026-07-02 14:58:53 Request is submitted
2026-07-02 14:58:54 Request is active
2026-07-02 14:59:19 Calling 'nice mars /tmp/20260702-1250/c9/tmp-_mars-s4tfsW-07e9853b44cd7f0accc7aa06c5ee9d8e.req'
2026-07-02 14:59:19 Forcing MIR_CACHE_PATH=/data/ec_coeff
2026-07-02 14:59:19 mars - WARN -
2026-07-02 14:59:19 mars - WARN -
2026-07-02 14:59:19 MIR environment variables:
2026-07-02 14:59:19 MIR_CACHE_PATH=/data/ec_coeff
2026-07-02 14:59:19 MIR_LSM_NAMED=1km.climate.v013
2026-07-02 14:59:19 Using MARS binary: /usr/local/apps/mars/versions/6.34.7.0/bin/mars.bin
2026-07-02 14:59:19 mars - INFO   - 20260702.1258

In [10]:
raw_soil

{'swvl1': array([[-4.70504165e-06, -4.70504165e-06, -4.70504165e-06, ...,
          2.09498469e-01,  1.88609187e-01,  1.64942805e-01],
        [-4.70504165e-06, -4.70504165e-06, -4.70504165e-06, ...,
          2.09498469e-01,  1.88609187e-01,  1.64942805e-01]]),
 'swvl2': array([[3.90410423e-06, 3.90410423e-06, 3.90410423e-06, ...,
         1.72901243e-01, 1.51798338e-01, 1.48822874e-01],
        [3.90410423e-06, 3.90410423e-06, 3.90410423e-06, ...,
         1.72901243e-01, 1.51798338e-01, 1.48822874e-01]]),
 'stl1': array([[265.86303711, 265.92358398, 265.96069336, ..., 220.19116211,
         220.67358398, 221.11889648],
        [265.86303711, 265.92358398, 265.96069336, ..., 220.19116211,
         220.67358398, 221.11889648]]),
 'stl2': array([[266.97900391, 267.00634766, 267.04345703, ..., 220.29345703,
         220.77197266, 221.21337891],
        [266.97900391, 267.00634766, 267.04345703, ..., 220.29345703,
         220.77197266, 221.21337891]])}

## 9. Transformaciones requeridas

Después de la lectura desde MARS se mantienen transformaciones de compatibilidad con el estado de entrada que espera AIFS Single v2:

1. `mwd` se convierte a `cos_mwd` y `sin_mwd`;
2. `sot` y `vsw` se renombran a `stl*` y `swvl*`;
3. `q_10` y `q_50` se retiran;
4. nieve y humedad del suelo se enmascaran sobre océano;
5. `gh` se convierte a geopotencial `z` con \(g_0=9.80665\ \mathrm{m\,s^{-2}}\).

Estas transformaciones no son exclusivas de ECMWF Open Data. Son necesarias si la recuperación MARS usa los nombres `gh`, `sot` y `vsw`, porque el runner de AIFS espera el convenio final de nombres y unidades.


In [13]:
SOIL_RENAME = {
    "sot_1": "stl1",
    "sot_2": "stl2",
    "vsw_1": "swvl1",
    "vsw_2": "swvl2",
}

SOIL_DIRECT = ("stl1", "stl2", "swvl1", "swvl2")

STANDARD_GRAVITY = 9.80665


def _merge_soil_fields(
    transformed: dict[str, np.ndarray],
    soil: Mapping[str, np.ndarray],
) -> None:
    """
    Inserta los campos de suelo en `transformed`.

    Soporta dos convenios:
    1. MARS soil-level: sot_1, sot_2, vsw_1, vsw_2.
    2. MARS single-level: stl1, stl2, swvl1, swvl2.

    El segundo convenio es el que suele estar disponible para análisis
    operativos como campos de superficie/single-level.
    """

    if all(name in soil for name in SOIL_RENAME):
        for source_name, target_name in SOIL_RENAME.items():
            transformed[target_name] = soil[source_name]
        return

    if all(name in soil for name in SOIL_DIRECT):
        for name in SOIL_DIRECT:
            transformed[name] = soil[name]
        return

    expected_options = sorted(set(SOIL_RENAME) | set(SOIL_DIRECT))
    raise KeyError(
        "No se reconoció el convenio de suelo. "
        f"Campos disponibles: {sorted(soil)}. "
        f"Se esperaba uno de estos conjuntos: {expected_options}."
    )


def transform_initial_conditions(
    fields: Mapping[str, np.ndarray],
    soil: Mapping[str, np.ndarray],
) -> dict[str, np.ndarray]:
    """
    Adapta los campos meteorológicos descargados al formato esperado por AIFS.

    La función transforma la dirección media del oleaje ``mwd`` en sus
    componentes ``cos_mwd`` y ``sin_mwd``, incorpora las variables de suelo
    según el convenio disponible, elimina variables no requeridas por el modelo
    y convierte la altura geopotencial ``gh`` a geopotencial ``z``.

    También aplica una máscara oceánica a variables de superficie terrestre,
    asignando ``NaN`` sobre puntos oceánicos.
    """

    transformed = dict(fields)

    mwd = transformed.pop("mwd")
    mwd_rad = np.deg2rad(mwd)
    transformed["cos_mwd"] = np.cos(mwd_rad)
    transformed["sin_mwd"] = np.sin(mwd_rad)

    _merge_soil_fields(transformed, soil)

    transformed.pop("q_10", None)
    transformed.pop("q_50", None)

    ocean_mask = transformed["lsm"][0] < 0.5

    for name in ("sd", "swvl1", "swvl2"):
        values = transformed[name].copy()
        values[:, ocean_mask] = np.nan
        transformed[name] = values

    if 'gh' in LEVELS:
        for level in LEVELS:
            gh = transformed.pop(f"gh_{level}")
            transformed[f"z_{level}"] = gh * STANDARD_GRAVITY

    return transformed


In [14]:
model_fields = None

if raw_fields is not None and raw_soil is not None:
    model_fields = transform_initial_conditions(
        raw_fields,
        raw_soil,
    )
    print("Campos transformados:", len(model_fields))
else:
    print("Transformación omitida.")

Campos transformados: 97


## 10. Validación estructural

Antes de cargar el checkpoint se comprueban nombres, dos tiempos, forma común y fracción de datos finitos.

In [15]:
def expected_model_field_names() -> set[str]:
    """
    Construye el conjunto de nombres de campos meteorológicos esperados por
    el modelo.

    Incluye variables de superficie, oleaje, suelo y variables atmosféricas
    en niveles de presión. La dirección media del oleaje ``mwd`` se reemplaza
    por sus componentes trigonométricas ``sin_mwd`` y ``cos_mwd``. Para cada
    nivel de presión se agregan geopotencial, temperatura y componentes del
    viento; la humedad específica se incluye excepto en los niveles 10 y
    50 hPa.

    Retorna
    -------
    set[str]
        Conjunto con los nombres de variables que deben estar presentes en
        los campos de entrada del modelo.
    """

    expected = set(PARAM_SFC)

    expected.update(PARAM_WAVE)
    expected.remove("mwd")
    expected.update({"sin_mwd", "cos_mwd"})

    expected.update(SOIL_RENAME.values())

    for level in LEVELS:
        expected.update(
            {
                f"z_{level}",
                f"t_{level}",
                f"u_{level}",
                f"v_{level}",
            }
        )
        if level not in {10, 50}:
            expected.add(f"q_{level}")

    return expected


EXPECTED_FIELDS = expected_model_field_names()
print("Número esperado de campos:", len(EXPECTED_FIELDS))

Número esperado de campos: 97


In [16]:
def validate_model_fields(
    fields: Mapping[str, np.ndarray],
) -> pd.DataFrame:
    """
    Valida la estructura, consistencia y calidad básica de los campos de
    entrada requeridos por el modelo.

    Esta función revisa que el diccionario de campos meteorológicos contenga
    exactamente las variables esperadas por el flujo de inferencia. Primero
    verifica que no falten variables obligatorias y que no existan variables
    adicionales no contempladas en ``EXPECTED_FIELDS``. Posteriormente,
    inspecciona la forma, el tipo de dato y la calidad numérica básica de
    cada campo.

    Se asume que cada variable está representada como un arreglo bidimensional
    con dimensiones ``(tiempo, espacio)``. Además, la función verifica que todas 
    las variables tengan el mismo tamaño espacial. 

    Parámetros
    ----------
    fields : Mapping[str, np.ndarray]
        Diccionario o estructura tipo diccionario que contiene los campos
        meteorológicos de entrada. Las claves corresponden a los nombres de
        las variables.

    Retorna
    -------
    pd.DataFrame
        Tabla resumen con una fila por variable. 

    Raises
    ------
    KeyError
        Se lanza si falta alguna variable definida en ``EXPECTED_FIELDS`` o si
        existen variables no esperadas dentro de ``fields``.

    ValueError
        Se lanza si alguna variable no tiene dos dimensiones, si la dimensión
        temporal no tiene longitud 2 o si el tamaño espacial no es consistente
        entre variables.
    """

    missing = EXPECTED_FIELDS - set(fields)
    unexpected = set(fields) - EXPECTED_FIELDS

    if missing:
        raise KeyError(f"Faltan campos: {sorted(missing)}")
    if unexpected:
        raise KeyError(f"Campos no esperados: {sorted(unexpected)}")

    rows = []
    spatial_size = None

    for name in sorted(fields):
        values = np.asarray(fields[name])

        if values.ndim != 2:
            raise ValueError(
                f"{name}: forma esperada (tiempo, espacio); "
                f"forma recibida {values.shape}."
            )

        if values.shape[0] != 2:
            raise ValueError(
                f"{name}: se esperaban 2 tiempos y se recibieron "
                f"{values.shape[0]}."
            )

        if spatial_size is None:
            spatial_size = values.shape[1]
        elif values.shape[1] != spatial_size:
            raise ValueError(
                f"{name}: tamaño espacial inconsistente."
            )

        rows.append(
            {
                "variable": name,
                "shape": str(values.shape),
                "dtype": str(values.dtype),
                "finite_fraction": float(np.isfinite(values).mean()),
                "min": float(np.nanmin(values)),
                "max": float(np.nanmax(values)),
            }
        )

    return pd.DataFrame(rows).set_index("variable")


validation_report = None

if model_fields is not None:
    validation_report = validate_model_fields(model_fields)
    display(validation_report.head(15))
    print("Validación estructural completada.")
else:
    print(
        "Validación de datos omitida. Se esperan "
        f"{len(EXPECTED_FIELDS)} campos."
    )

,shape,dtype,finite_fraction,min,max
variable,,,,,
10u,"(2, 542080)",float64,1.000000,-21.624603,20.671295
10v,"(2, 542080)",float64,1.000000,-18.013168,20.589371
2d,"(2, 542080)",float64,1.000000,207.669586,301.021149
2t,"(2, 542080)",float64,1.000000,210.481400,315.651321
cdww,"(2, 542080)",float64,0.708445,0.000949,0.003261
cos_mwd,"(2, 542080)",float64,0.664328,-1.000000,1.000000
h1012,"(2, 542080)",float64,0.664328,0.002004,4.301077
h1214,"(2, 542080)",float64,0.664328,0.001357,3.842910
h1417,"(2, 542080)",float64,0.664328,0.001578,4.008780


Validación estructural completada.


## 11. Construcción de `input_state`

In [17]:
input_state = None

if model_fields is not None:
    input_state = {
        "date": DATE,
        "fields": model_fields,
    }
    print("Fecha:", input_state["date"])
    print("Campos:", len(input_state["fields"]))
else:
    print("input_state no se creó.")

Fecha: 2020-10-05 18:00:00
Campos: 97


## 12. Carga del modelo

El cambio esencial de checkpoint es:

```python
{"huggingface": "ecmwf/aifs-single-2.0"}
```

In [18]:
CHECKPOINT = {"huggingface": "ecmwf/aifs-single-2.0"}
runner = None

if CONFIG.run_inference:
    if input_state is None:
        raise RuntimeError(
            "No existe input_state. Activa la descarga y ejecuta "
            "las secciones anteriores."
        )

    runner = SimpleRunner(
        CHECKPOINT,             # AIFS V2.0  
        device=CONFIG.device,   # cuDA
    )
    print("Runner creado.")
else:
    print("Carga del modelo omitida.")

Runner creado.


## 13. Inferencia corta

Con `lead_time_hours=6` se espera un estado válido en \(t_0+6\) h. En producción convendrá procesar cada estado incrementalmente.

In [ ]:
forecast_states: list[dict] = []

if runner is not None:
    # Ejecuta el modelo de pronóstico a partir del estado inicial.
    # runner.run(...) devuelve una secuencia de estados pronosticados
    # para distintos tiempos de adelanto.
    for state in runner.run(
        input_state=input_state,              # Estado atmosférico inicial del forecast
        lead_time=CONFIG.lead_time_hours,     # Horizonte total del pronóstico en horas
    ):
        forecast_states.append(state)
        print_state(state)
    # Calcula cuántos estados de pronóstico se esperan.
    expected_states = CONFIG.lead_time_hours // 6

    # Verifica que el número de estados generados coincida con el esperado.
    if len(forecast_states) != expected_states:
        raise RuntimeError(
            f"Se esperaban {expected_states} estados y se "
            f"obtuvieron {len(forecast_states)}."
        )

    # Si la validación fue exitosa, imprime el número de estados generados.
    print("Estados generados:", len(forecast_states))

else:
    print("Inferencia omitida.")

## 14. Inspección mínima de salida

In [ ]:
def summarize_forecast_state(
    state: Mapping,
) -> pd.DataFrame:
    """
    Genera un resumen tabular de las variables contenidas en un estado
    de pronóstico.

    Parámetros
    ----------
    state : Mapping
        Diccionario o estructura tipo diccionario que contiene la información
        del estado atmosférico. Se espera que incluya una clave "fields",
        donde cada entrada corresponde a una variable meteorológica.

    Retorna
    -------
    pd.DataFrame
        Tabla con una fila por variable. Incluye el nombre de la variable,
        la forma del arreglo, el tipo de dato y la fracción de valores finitos.
    """

    rows = []
    for name, values in sorted(state["fields"].items()):
        array = np.asarray(values)
        rows.append(
            {
                # Nombre de la variable, por ejemplo: temperatura,
                "variable": name,

                # Dimensiones del arreglo.
                "shape": str(array.shape),

                # Tipo de dato numérico.
                "dtype": str(array.dtype),

                # Fracción de valores finitos en el arreglo.
                # Un valor de 1.0 indica que todos los datos son finitos.
                # Valores menores a 1.0 indican presencia de NaN, inf o -inf.
                "finite_fraction": float(np.isfinite(array).mean()),
            }
        )
    return pd.DataFrame(rows).set_index("variable")


if forecast_states:
    last_state = forecast_states[-1]
    output_summary = summarize_forecast_state(last_state)

    # Selecciona únicamente las variables de interés definidas en CONFIG.output_variables.
    selected = output_summary.index.intersection(
        CONFIG.output_variables
    )
    display(output_summary.loc[selected])
    print("Fecha válida:", last_state["date"])
else:
    last_state = None
    print("No hay estados para inspeccionar.")

## 15. Interpolación N320 → 0.25° y recorte

Solo se procesan unas pocas variables y el último horizonte. Esto permite validar el posprocesamiento sin escribir un archivo global enorme.

In [ ]:
UNITS = {
    "msl": "Pa",
    "10u": "m s-1",
    "10v": "m s-1",
    "t_850": "K",
    "q_850": "kg kg-1",
    "z_500": "m2 s-2",
}


def n320_to_regular(values: np.ndarray) -> np.ndarray:
    """
    Interpola un campo meteorológico desde la grilla reducida gaussiana N320
    hacia una grilla regular latitud-longitud de 0.25° x 0.25°.

    Parámetros
    ----------
    values : np.ndarray
        Arreglo con los valores de una variable meteorológica definida sobre
        la grilla N320.

    Retorna
    -------
    np.ndarray
        Arreglo interpolado sobre una grilla regular global de 0.25° de
        resolución espacial. La forma esperada del arreglo resultante es
        ``(721, 1440)``, correspondiente a 721 latitudes y 1440 longitudes.

    Raises
    ------
    ValueError
        Se lanza si la interpolación no produce un arreglo con la forma
        esperada ``(721, 1440)``.
    """
    regular = ekr.interpolate(
        values,
        {"grid": "N320"},
        {"grid": (0.25, 0.25)},
    )

    if regular.shape != (721, 1440):
        raise ValueError(
            f"Forma regular inesperada: {regular.shape}"
        )

    return regular


def subset_regular_grid(
    regular_values: np.ndarray,
    *,
    variable_name: str,
    config: NotebookConfig,
) -> xr.DataArray:
    """
    Convierte un campo global en grilla regular de 0.25° x 0.25° a un
    ``xarray.DataArray`` y extrae el subconjunto regional definido en la
    configuración del notebook.

    Parámetros
    ----------
    regular_values : np.ndarray
        Arreglo bidimensional con los valores de una variable meteorológica
        ya interpolada a una grid regular global de 0.25° x 0.25°.

    variable_name : str
        Nombre de la variable meteorológica. 

    config : NotebookConfig
        Objeto de configuración del notebook. Debe contener los límites
        espaciales de la región de interés mediante los atributos
        ``lat_min``, ``lat_max``, ``lon_min`` y ``lon_max``.

    Retorna
    -------
    xr.DataArray
        Campo meteorológico regional como ``xarray.DataArray``, con
        dimensiones ``latitude`` y ``longitude``, coordenadas geográficas
        explícitas y atributos descriptivos de unidades, grid de origen
        y grid de destino.
    """

    latitudes = np.arange(
        90.0, -90.25, -0.25, dtype=np.float32
    )
    longitudes = np.arange(
        0.0, 360.0, 0.25, dtype=np.float32
    )

    data_array = xr.DataArray(
        regular_values,
        dims=("latitude", "longitude"),
        coords={
            "latitude": latitudes,
            "longitude": longitudes,
        },
        name=variable_name,
        attrs={
            "units": UNITS.get(variable_name, "unknown"),
            "source_grid": "N320",
            "target_grid": "0.25 degree regular lat-lon",
        },
    )

    return data_array.sel(
        latitude=slice(config.lat_max, config.lat_min),
        longitude=slice(config.lon_min, config.lon_max),
    )


def state_to_regional_dataset(
    state: Mapping,
    *,
    variables: Iterable[str],
    config: NotebookConfig,
    initial_date: dt.datetime,
) -> xr.Dataset:
    """
    Convierte un estado de pronóstico del modelo AIFS en un ``xarray.Dataset``
    regional con variables meteorológicas seleccionadas.

    El conjunto de datos resultante incluye una dimensión temporal llamada
    ``valid_time``, asociada con la fecha válida del estado de pronóstico.
    Además, se añaden atributos globales que documentan el modelo utilizado,
    la fecha inicial del pronóstico y el intervalo temporal entre salidas.

    Parámetros
    ----------
    state : Mapping
        Estado de pronóstico generado por el modelo. 

    variables : Iterable[str]
        Lista o iterable con los nombres de las variables que se desean
        extraer del estado de pronóstico.

    config : NotebookConfig
        Objeto de configuración del notebook. Define los límites espaciales
        del subconjunto regional mediante ``lat_min``, ``lat_max``,
        ``lon_min`` y ``lon_max``.

    initial_date : dt.datetime
        Fecha inicial del pronóstico. 

    Retorna
    -------
    xr.Dataset
        Conjunto de datos regional con las variables meteorológicas
        seleccionadas, interpoladas a una grilla regular de 0.25° x 0.25°,
        recortadas al dominio de interés y organizadas con una dimensión
        temporal ``valid_time``.

    Raises
    ------
    KeyError
        Se lanza si alguna de las variables solicitadas no está presente en
        ``state["fields"]``.

    ValueError
        Puede propagarse desde ``n320_to_regular`` si la interpolación no
        genera un arreglo con la forma esperada.
    """

    data_vars = {}

    for name in variables:
        if name not in state["fields"]:
            raise KeyError(
                f"La variable {name!r} no está en la salida."
            )

        regular = n320_to_regular(
            np.asarray(state["fields"][name])
        )

        data_vars[name] = subset_regular_grid(
            regular,
            variable_name=name,
            config=config,
        )

    dataset = xr.Dataset(data_vars)
    dataset = dataset.expand_dims(
        valid_time=[np.datetime64(state["date"])]
    )

    dataset.attrs.update(
        {
            "title": "AIFS Single v2 minimal validation output",
            "model": "ecmwf/aifs-single-2.0",
            "initial_time": initial_date.isoformat(),
            "forecast_step_hours": 6,
        }
    )

    return dataset

In [ ]:
regional_dataset = None

if last_state is not None:
    regional_dataset = state_to_regional_dataset(
        last_state,
        variables=CONFIG.output_variables,
        config=CONFIG,
        initial_date=DATE,
    )
    display(regional_dataset)
else:
    print("Posprocesamiento omitido.")

## 16. Gráfica opcional

In [ ]:
if regional_dataset is not None:

    import matplotlib.pyplot as plt
    import cartopy.crs as ccrs
    import cartopy.feature as cfeature

    field_to_plot = "msl"                                   #VARIABLE A PLOTEAR
    plot_data = regional_dataset[field_to_plot].isel(
        valid_time=0
    )

    if plot_data.attrs.get("units") == "Pa":
        plot_data = plot_data / 100.0
        plot_data.attrs["units"] = "hPa"


    #INICIO DE LA FIGURA RECORTADA
    
    figure = plt.figure(figsize=(11, 6))
    axis = plt.axes(projection=ccrs.PlateCarree())

    plot_data.plot(
        ax=axis,
        transform=ccrs.PlateCarree(),
        x="longitude",
        y="latitude",
    )

    axis.coastlines()
    axis.add_feature(cfeature.BORDERS, linestyle=":")
    axis.set_title(
        f"{field_to_plot} | AIFS Single v2 | "
        f"{pd.Timestamp(last_state['date']).isoformat()}"
    )

    plt.show()
else:
    print("Gráfica omitida.")

## 17. Guardado del resultado regional

In [ ]:
if regional_dataset is not None:
    filename = (
        "aifs-single-v2_"
        f"{DATE:%Y%m%dT%H%M}_"
        f"lead-{CONFIG.lead_time_hours:03d}h_"
        "regional-validation.nc"
    )
    output_file = CONFIG.output_dir / filename

    encoding = {
        name: {
            "zlib": True,
            "complevel": 4,
            "dtype": "float32",
        }
        for name in regional_dataset.data_vars
    }

    regional_dataset.to_netcdf(
        output_file,
        encoding=encoding,
    )

    print("Guardado en:", output_file.resolve())
else:
    print("Guardado omitido.")

## ES NECESARIO AGREGAR METADATA MAS COMPLETA PARA LOS PRONOSTICOS